In [1]:
import os
import pickle
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, desc, asc
from pyspark.sql.types import FloatType

# Evaluation


/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
# ==============================================================================
# PARAMÈTRES EXIGÉS PAR LE PROJET (Deliverable 2)
# ==============================================================================
query = "Generate exam questions on Big Data frameworks"
embedding_strategy = "tfidf"                # Choix: "sbert", "word2vec", "tfidf"
similarity_strategy = "cosine"              # Choix: "cosine", "euclidean"
number_of_text_chunks_to_retrieve = 4
number_of_questions_to_generate = 2
generator_backend = "ollama"                
generator_model = "llama3.2:3b"
max_context_tokens = 2048                   # Limite de sécurité pour l'IA
input_dir = "/home/guest/shared/course_material/"
output_dir = "/home/guest/shared/course_material/output/"

print("✅ Paramètres du système RAG chargés.")

✅ Paramètres du système RAG chargés.


In [3]:
# ==============================================================================
# CELLULE 1 : INITIALISATION SPARK ET FONCTIONS DE DISTANCE (UDFS)
# ==============================================================================
# 1. Démarrage de la session Spark (Le "Driver")
spark = SparkSession.builder \
    .appName("RAG_Retrieval_Engine") \
    .master("local[*]") \
    .getOrCreate()

# Désactivation du parallélisme interne d'HuggingFace pour éviter les conflits avec Spark
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 17:33:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/21 17:33:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/21 17:33:17 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
# 2. Fonctions mathématiques distribuées (UDFs)
# L'utilisation d'UDFs permet d'exécuter le calcul sur tous les noeuds Spark
# en parallèle (O(N) distribué), évitant un .collect() prématuré de toute la base.

def make_spark_cosine_udf(query_vector):
    """Calcule la similarité Cosinus (Plus proche de 1 = Meilleur)."""
    def compute_cosine(doc_emb):
        if doc_emb is None: return 0.0
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        
        norm_q = np.linalg.norm(q)
        norm_d = np.linalg.norm(d)
        if norm_q == 0 or norm_d == 0: return 0.0
            
        return float(np.dot(q, d) / (norm_q * norm_d))
    return udf(compute_cosine, FloatType())

def make_spark_euclidean_udf(query_vector):
    """Calcule la distance Euclidienne (Plus proche de 0 = Meilleur)."""
    def compute_euclidean(doc_emb):
        if doc_emb is None: return 999.0 # Valeur arbitraire haute pour repousser les erreurs
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        
        # Formule de la distance euclidienne : racine de la somme des carrés des différences
        return float(np.linalg.norm(q - d))
    return udf(compute_euclidean, FloatType())

print("✅ Moteur Spark initialisé et UDFs mathématiques prêtes.")

✅ Moteur Spark initialisé et UDFs mathématiques prêtes.


In [5]:
# ==============================================================================
# CELLULE 2 : FONCTION DE RETRIEVAL (O(N) distribué)
# ==============================================================================

def retrieve_top_chunks(query: str, embedding_strategy: str, similarity_metric: str, k: int = 4):
    """
    Recherche les K meilleurs paragraphes selon la stratégie et la métrique choisies.
    """
    print(f"🔍 Recherche en cours... [Stratégie: {embedding_strategy} | Métrique: {similarity_metric}]")
    
    # 1. Chargement dynamique de la base de données Parquet appropriée
    parquet_path = f"/home/guest/shared/course_material/embeddings_{embedding_strategy}.parquet"
    
    try:
        df_corpus = spark.read.parquet(parquet_path)
    except Exception as e:
        print(f"❌ Erreur : Impossible de lire le fichier Parquet pour {embedding_strategy}.")
        return []

    # 2. Vectorisation de la question utilisateur (Doit matcher la stratégie d'entraînement)
    if embedding_strategy == "sbert":
        encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        q_vec = encoder.encode(query, normalize_embeddings=True)
        
    elif embedding_strategy == "word2vec":
        from pyspark.ml.feature import Word2VecModel, Tokenizer
        
        # 1. Spark MLlib a besoin d'un mini DataFrame pour transformer la question
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])
        
        # 2. On découpe la question en mots (exactement comme dans la Tâche 1)
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        wordsData = tokenizer.transform(df_query)
        
        # 3. On charge le modèle Word2Vec de SPARK (il comprend très bien les dossiers !)
        w2v_model = Word2VecModel.load("/home/guest/shared/course_material/w2v_model.model")
        
        # 4. On calcule le vecteur de la question
        w2vData = w2v_model.transform(wordsData)
        
        # 5. On extrait le vecteur final sous forme de tableau NumPy
        q_vec = w2vData.select("features").first()[0].toArray()
        
    elif embedding_strategy == "tfidf":
        from pyspark.ml.feature import IDFModel, Tokenizer, HashingTF
        
        # Spark MLlib a besoin d'un mini DataFrame pour transformer la question
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])
        
        # On repasse la question dans les mêmes filtres que la Tâche 1
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        wordsData = tokenizer.transform(df_query)
        
        hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000)
        featurizedData = hashingTF.transform(wordsData)
        
        # On charge ton cerveau sauvegardé
        idfModel = IDFModel.load("/home/guest/shared/course_material/tfidf_model.model")
        rescaledData = idfModel.transform(featurizedData)
        
        # On extrait le vecteur final (on le convertit en array classique pour le calcul)
        q_vec = rescaledData.select("features").first()[0].toArray()
    
    else:
        raise ValueError(f"Stratégie inconnue : {embedding_strategy}")

    # 3. Application de la bonne métrique mathématique et tri
    if similarity_metric == "cosine":
        # Cosinus : On trie par ordre décroissant (le plus grand score gagne)
        metric_udf = make_spark_cosine_udf(q_vec)
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding")))
        df_top_k = df_scored.orderBy(desc("score")).limit(k)
        
    elif similarity_metric == "euclidean":
        # Euclidienne : On trie par ordre croissant (la plus petite distance gagne)
        metric_udf = make_spark_euclidean_udf(q_vec)
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding")))
        df_top_k = df_scored.orderBy(asc("score")).limit(k)
        
    else:
        raise ValueError(f"Métrique inconnue : {similarity_metric}")
    
    # 4. Rapatriement local des K résultats UNIQUEMENT (Conforme aux consignes)
    return df_top_k.collect()

In [6]:

# --- TEST DE LA FONCTION ---
ma_question = "What is a Neural Network?"

# On teste avec SBERT et Cosinus (Tu peux changer les paramètres pour tester Euclidienne)
resultats = retrieve_top_chunks(
    query=query, 
    embedding_strategy=embedding_strategy, 
    similarity_metric=similarity_strategy, 
    k=number_of_text_chunks_to_retrieve
)

for i, row in enumerate(resultats, 1):
    print(f"TOP {i} [Score: {row['score']:.4f}] - Doc: {row['source_pdf']} (p.{row['page_num']})")

🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


TOP 1 [Score: 0.2535] - Doc: mr.pdf (p.94)
TOP 2 [Score: 0.1781] - Doc: a_TowardScalableSystemsForBigDataAnalytics.pdf (p.3)
TOP 3 [Score: 0.1729] - Doc: a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf (p.9)
TOP 4 [Score: 0.1633] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.210)


## Interprêtation

A la###  question = "What is a Neural Netwo
rk?"

- score Sbert
    - TOP 1 [Score: 0.6328] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
    - TOP 2 [Score: 0.5829] - Doc: A Neural Network Playground.pdf (p.1)
    - TOP 3 [Score: 0.5711] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
    - TOP 4 [Score: 0.5706] - Doc: Intro_DL_H515_2026.pdf (p.25)


- score word2vec
    - TOP 1 [Score: 0.1938] - Doc: 1-intro.pdf (p.2)
    - TOP 2 [Score: 0.1741] - Doc: intro.pdf (p.3)
    - TOP 3 [Score: 0.1694] - Doc: 1-intro.pdf (p.19)
    - TOP 4 [Score: 0.1607] - Doc: 1-intro.pdf (p.30)


- score tfidf
    - TOP 1 [Score: 0.1719] - Doc: Intro_DL_H515_2026.pdf (p.6)
    - TOP 2 [Score: 0.1466] - Doc: A Neural Network Playground.pdf (p.1)
    - TOP 3 [Score: 0.1329] - Doc: 1-intro.pdf (p.3)
    - TOP 4 [Score: 0.1306] - Doc: 5-nosql.pdf (p.34) 

### Conclusion:

L'expérience montre que Word2Vec produit des scores de similarité cosinus très bas (~0.18) comparé à SBERT (~0.60).
Cela s'explique par l'approche 'Bag-of-Words' (moyenne des vecteurs) de Word2Vec qui dilue l'information avec les mots de 
liaison de la requête, et par la taille restreinte de notre corpus d'entraînement local par rapport au modèle SBERT 
pré-entraîné.


SBERT produit des scores hauts (~0.60) car il crée des vecteurs denses (384 dimensions) qui capturent le sens global, même
si les mots exacts diffèrent.
TF-IDF produit des scores bas (~0.17) car il génère des vecteurs creux (sparses) extrêmement grands où seuls les mots-clés
exacts créent une intersection mathématique.

In [7]:
# ==============================================================================
# CELLULE 3 : GÉNÉRATION DES QUESTIONS D'EXAMEN (LLM LOCAL)
# ==============================================================================

def generate_exam_with_llm(query: str, spark_results, num_questions: int = 3):
    """
    Construit un prompt sourcé et interroge Ollama.
    """
    if not spark_results:
        return "Aucun document n'a été trouvé pour générer l'examen."

    # 1. Construction du contexte avec les métadonnées (Traçabilité exigée)
    context_text = ""
    for idx, row in enumerate(spark_results, 1):
        context_text += f"\n[{idx}] (Source_pdf: {row['source_pdf']}, Page {row['page_num']}, ID: {row['chunk_id']})\n{row['chunk_text']}\n"
    
    # 2. Ingénierie du Prompt (Prompt strict pour limiter les hallucinations)
    prompt = f"""You are a strict university teaching assistant designing an exam.
TASK: Generate {num_questions} exam questions about the topic: "{query}".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:
{context_text}
"""
    
    # EXIGENCE DU PROJET : Afficher les paramètres et le prompt exact
    print("\n" + "-"*70)
    print("⚙️ PARAMÈTRES DE GÉNÉRATION UTILISÉS :")
    print(f"- Backend : {generator_backend}")
    print(f"- Modèle : {generator_model}")
    print(f"- Max context tokens : {max_context_tokens}")
    print("-" * 70)
    print("📝 PROMPT EXACT ENVOYÉ À L'IA :")
    print(prompt)
    print("-" * 70)
    # 3. Appel de l'API Ollama
    print("\n⏳ Envoi du prompt à Ollama (modèle local)...")
    try:
        response = requests.post(
            "http://host.docker.internal:11434/api/generate",
            json={
                "model": "llama3.2:3b",
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]
    
    except requests.exceptions.RequestException as e:
        return f"Erreur de connexion à l'API Ollama.{e}"

# --- GÉNÉRATION FINALE ---
examen_genere = generate_exam_with_llm(
    query=query, 
    spark_results=resultats, 
    num_questions=number_of_questions_to_generate
)

print("\n" + "="*70)
print("🎓 EXAMEN GÉNÉRÉ PAR L'IA :")
print("="*70)
print(examen_genere)


----------------------------------------------------------------------
⚙️ PARAMÈTRES DE GÉNÉRATION UTILISÉS :
- Backend : ollama
- Modèle : llama3.2:3b
- Max context tokens : 2048
----------------------------------------------------------------------
📝 PROMPT EXACT ENVOYÉ À L'IA :
You are a strict university teaching assistant designing an exam.
TASK: Generate 2 exam questions about the topic: "Generate exam questions on Big Data frameworks".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:

[1] (Source_pdf: mr.pdf, Page 94, ID: p94_c0)
exam question 94 / 103

[2] (Source_pdf: a_TowardScalableSystemsForBigDataAnalytics.pdf, Page 3, ID: p3_c3)
metric. However, i

# Task 3 : Evaluation 

In [8]:
import re
import json
import time
import psutil
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional

## build a small evaluation set of representative course-related queries
- Pour construire cet ensemble, on survole les slides, et ont 

In [9]:
# Gold set for 1-intro.pdf
# INFO-H515 Lecture 1: Introduction, HDFS, MapReduce
GOLD_SET = [
    {
    "query": "What is the difference between a transformation and an action in Spark?", 
    "relevant_pages": {23, 24, 25, 36, 37},
    "expected_topic": "Spark / Distributed Big Data Course (Transformation, action)",
    "expected_answer_elements":[
        "transformation, RDD, immediately executing computation",
        "An action triggers execution and produces a result or output"
    ]
},
{
    "query": "Why is lazy evaluation important in Spark?", 
    "relevant_pages": {63, 64, 65},
    "expected_topic": "Spark / Distributed Big Data Course (lazy evaluation)",
    "expected_answer_elements":[
        "Lazy evaluation, delay, distributed processing, efficiency",
    ]
},
{
    "query": "What is shuffle in Spark and why is it expensive?", 
    "relevant_pages": {28, 29, 67, 68},
    "expected_topic": "Spark / Distributed Big Data Course (shuffle)",
    "expected_answer_elements":[
        "redistribution, data, partitions, operations",
        "network, communication",
    ]
},
{
    "query": "What problem does Lambda Architecture solve?", 
    "relevant_pages": {14, 49},
    "expected_topic": "Streaming (Lambda Architecture)",
    "expected_answer_elements":[
        "conflict, accurate, batch processing",
        "latency",
    ]
},
{
    "query": "What is windowing in stream processing?", 
    "relevant_pages": {108, 109, 110},
    "expected_topic": "Streaming (windowing in stream)",
    "expected_answer_elements":[
        "restricts computation, time interval, entire stream",
    ]
},
{
    "query": "What is overfitting and why is it problematic?", 
    "relevant_pages": {17, 18, 19},
    "expected_topic": "Introduction to Big Data Analytics",
    "expected_answer_elements":[
        "model, noise, specific patterns, training data, generalizable relationships.",
        "predictive performance",
    ]
},
{
    "query": "What is bagging and why is it useful?", 
    "relevant_pages": {24, 25},
    "expected_topic": "Introduction to Big Data Analytics (bagging)",
    "expected_answer_elements":[
        "multiple models, bootstrap samples, training dataset,variance, stability",
    ]
},
{
    "query": "Why does lower perplexity not necessarily imply better factual correctness?", 
    "relevant_pages": {4,11},
    "expected_topic": "Large Language Models Course (LLM Lecture 2.pdf)",
    "expected_answer_elements":[
        "token, prediction, hallucinating, training metric",
    ]
},
]

## Retrieval metrics : precision@k / recall@k


### Generation and grounding metrics

In [10]:
#Définir le format de sortie attendu
# Format JSON attendu pour chaque génération
OUTPUT_SCHEMA = {
    "query": "str",
    "questions": [
        {
            "question": "str",
            "answer": "str", 
            "citations": ["pdf_name p.X chunk_id", ...]
        }
    ],
    "refusal": "str | null"  # si les sources sont insuffisantes
}

In [11]:
# Adapter les sources pour qu'elle match avec le format souhaité
def get_formatted_sources(retrieved: List[Dict]) -> str:
    sources = ""
    for i, r in enumerate(retrieved):
        sources += f"\n[{i+1}] (source: {r['source_pdf']}, page {r['page_num']}, chunk {r['chunk_id']})\n"
        sources += r["chunk_text"] + "\n"
    return sources

In [12]:
def build_exam_question_prompt(query: str, retrieved_chunks: list, n_questions: int = 3) -> str:
    prompt = f"""
TASK : Generate {n_questions} exam questions about {query}.
Use only the SOURCES below. If the sources are insufficient, say so. Cite sources using [1], [2],
etc. For each question, provide : Q, expected answer, and supporting citation(s).

SOURCES :
{get_formatted_sources(retrieved_chunks)}

OUTPUT FORMAT (strict JSON):
{OUTPUT_SCHEMA}
    """
    
    return prompt

In [13]:
# Appeler le LLM et parser la réponse
def generate_text_ollama(prompt: str, model: str = generator_model ) -> str:
    """Generate text using a local Ollama model.

    Requires Ollama running locally: https://ollama.com
    """
    url = "http://host.docker.internal:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,   # moins créatif = plus fidèle aux sources
            "num_predict": max_context_tokens,  # max tokens générés
        }
    }
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

In [14]:
# Pipeline complet pour une query

# Cette fonction prend le texte brut retourné par Ollama et essaie d'en extraire
# un objet JSON valide. On nettoie d'abord les backticks que le modèle ajoute
# parfois autour du JSON, puis on parse. Si ca ne marche pas, on retourne un
# objet vide mais bien formé pour que le reste du pipeline ne plante pas.
def parse_llm_output(raw_output: str) -> tuple:
    import re, json
    # On retire les blocs markdown ```json ... ``` si le modele les a ajoutes
    clean = re.sub(r"```(?:json)?", "", raw_output).strip().rstrip("`").strip()
    try:
        parsed = json.loads(clean)
        return parsed, True
    except json.JSONDecodeError:
        # Le modele n'a pas respecte le format JSON. On retourne une structure vide
        # et on signale l'echec pour que les metriques en tiennent compte.
        return {"query": "", "questions": [], "refusal": None}, False


def run_generation_pipeline(query: str, n_questions: int = 3) -> dict:
    """
    Pipeline complet pour une requete : retrieval Spark, construction du prompt,
    appel au LLM local, parsing de la reponse et collecte des metriques.
    """
    process = psutil.Process()
    
    # Etape 1 : on va chercher les k chunks les plus pertinents dans le corpus
    t0 = time.time()
    retrieved = retrieve_top_chunks(query, embedding_strategy, similarity_strategy, number_of_text_chunks_to_retrieve)
    retrieval_time = time.time() - t0
    
    # Etape 2 : on construit le prompt en injectant les chunks recuperes comme sources
    prompt = build_exam_question_prompt(query, retrieved, n_questions)
    # Approximation du nombre de tokens en entree (1 mot ~ 1 token)
    tokens_in = len(prompt.split())
    
    # Etape 3 : on mesure la memoire avant et apres la generation pour detecter les fuites
    mem_before = process.memory_info().rss / 1e6  # en MB
    t1 = time.time()
    raw_output = generate_text_ollama(prompt)
    generation_time = time.time() - t1
    mem_after = process.memory_info().rss / 1e6
    
    # Approximation du nombre de tokens en sortie
    tokens_out = len(raw_output.split())
    
    # Etape 4 : on essaie de parser la reponse du modele en JSON structure
    # parse_success vaut True si le JSON est valide, False sinon
    parsed_output, parse_success = parse_llm_output(raw_output)
    
    # On assemble toutes les informations dans un dictionnaire unique
    # pour faciliter l'evaluation et les comparaisons entre differentes configs
    return {
        # Parametres utilises pour cette requete
        "query":              query,
        "embedding_strategy": embedding_strategy,
        "similarity_metric":  similarity_strategy,
        "k":                  number_of_text_chunks_to_retrieve,
        "model":              generator_model,
        
        # Resultats du retrieval Spark
        "retrieved_chunks":   retrieved,
        "retrieval_time_s":   round(retrieval_time, 3),
    
        # Metriques LLM
        "prompt":             prompt,
        "raw_output":         raw_output,
        "tokens_in":          tokens_in,
        "tokens_out":         tokens_out,
        "generation_time_s":  round(generation_time, 3),
        "total_time_s":       round(retrieval_time + generation_time, 3),
        "memory_delta_mb":    round(mem_after - mem_before, 1),
        
        # Reponse du modele apres parsing
        "parsed_output":      parsed_output,
        "parse_success":      parse_success,
        "n_questions":        len(parsed_output.get("questions", [])),
        "refusal":            parsed_output.get("refusal"),
    }


In [15]:
# Lancer sur toutes les queries du gold set
results = []

for eval_item in GOLD_SET:
    print(f"\nQuery: {eval_item['query']}")
    result = run_generation_pipeline( query=eval_item["query"], n_questions=3)
    
    # Ajouter les gold labels pour l'évaluation
    result["gold_pages"] = eval_item["relevant_pages"]
    result["expected_topic"] = eval_item["expected_topic"]
    result["expected_answer_elements"] = eval_item["expected_answer_elements"]
    
    results.append(result)
    
print(f"\nTotal : {len(results)} queries traitées")


Query: What is the difference between a transformation and an action in Spark?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: Why is lazy evaluation important in Spark?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: What is shuffle in Spark and why is it expensive?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: What problem does Lambda Architecture solve?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: What is windowing in stream processing?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: What is overfitting and why is it problematic?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: What is bagging and why is it useful?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Query: Why does lower perplexity not necessarily imply better factual correctness?
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]



Total : 8 queries traitées


In [16]:
# Inspecter le premier résultat
r = results[0]
print(f"Query     : {r['query']}")
print(f"Tokens in : {r['tokens_in']}")
print(f"Tokens out: {r['tokens_out']}")
print(f"Time      : {r['total_time_s']}s")
print(f"Parsed    : {r['parse_success']}")
print()
for q in r["parsed_output"].get("questions", []):
    print(f"Q: {q['question']}")
    print(f"A: {q['answer'][:150]}...")
    print(f"C: {q['citations']}")
    print()

Query     : What is the difference between a transformation and an action in Spark?
Tokens in : 195
Tokens out: 158
Time      : 32.275s
Parsed    : False



In [17]:
r.keys()

dict_keys(['query', 'embedding_strategy', 'similarity_metric', 'k', 'model', 'retrieved_chunks', 'retrieval_time_s', 'prompt', 'raw_output', 'tokens_in', 'tokens_out', 'generation_time_s', 'total_time_s', 'memory_delta_mb', 'parsed_output', 'parse_success', 'n_questions', 'refusal', 'gold_pages', 'expected_topic', 'expected_answer_elements'])

In [23]:
# precision@k : parmi les k chunks recuperes, combien sont sur les bonnes pages ?
# Cette metrique evalue uniquement la qualite du retrieval Spark, pas la generation.
def precision_at_k(retrieved: list, relevant_pages: set, k: int) -> float:
    top = retrieved[:k]
    hits = sum(1 for r in top if r["page_num"] in relevant_pages)
    return hits / k if k > 0 else 0.0

# recall@k : parmi toutes les pages gold, combien ont ete recuperees ?
# Un recall faible signifie que le retrieval a rate des passages importants.
def recall_at_k(retrieved: list, relevant_pages: set, k: int) -> float:
    top = retrieved[:k]
    hit_pages = {r["page_num"] for r in top if r["page_num"] in relevant_pages}
    return len(hit_pages) / max(1, len(relevant_pages))


# grounding_score : est-ce que les mots-cles attendus apparaissent dans la reponse du LLM ?
# On cherche chaque element du gold set dans le texte de la reponse generee.
# C'est different du recall@k : on evalue la reponse, pas les chunks recuperes.
def compute_grounding(answer_text: str, expected_elements: list) -> float:
    if not expected_elements:
        return 1.0
    answer_lower = answer_text.lower()
    found = 0
    for elem in expected_elements:
        keywords = [kw.strip().lower() for kw in elem.split(",")]
        if any(kw in answer_lower for kw in keywords):
            found += 1
    return found / len(expected_elements)


# elements_coverage : version detaillee de grounding qui retourne aussi
# la liste des elements trouves et manquants pour faciliter le debug.
def compute_elements_coverage(answer_text: str, expected_elements: list) -> tuple:
    if not expected_elements:
        return 1.0, [], []
    answer_lower = answer_text.lower()
    found, missing = [], []
    for elem in expected_elements:
        keywords = [kw.strip().lower() for kw in elem.split(",")]
        if any(kw in answer_lower for kw in keywords):
            found.append(elem)
        else:
            missing.append(elem)
    return len(found) / len(expected_elements), found, missing


# citation_accuracy : les citations dans la reponse du LLM pointent-elles
# vers des sources qui ont reellement ete recuperees par Spark ?
# On croise les strings de citation avec les noms de PDF et chunk_id des chunks recuperes.
def check_citations(citations: list, retrieved: list) -> tuple:
    if not citations:
        return 0.0, 0, 0
    retrieved_pdfs = {r["source_pdf"].lower() for r in retrieved}
    retrieved_ids  = {r["chunk_id"].lower() for r in retrieved}
    valid = 0
    for cit in citations:
        cit_str = str(cit).lower()
        if any(pdf in cit_str for pdf in retrieved_pdfs) or \
           any(cid in cit_str for cid in retrieved_ids):
            valid += 1
    return valid / len(citations), valid, len(citations)


def evaluate_generation(result: dict) -> dict:
    """
    Evalue la qualite du pipeline pour une requete donnee.

    On distingue deux niveaux :
      - Retrieval  : precision@k et recall@k mesurent si Spark a recupere les bonnes pages.
      - Generation : grounding, citation_accuracy et elements_coverage mesurent
                     si le LLM a produit une reponse correcte a partir de ces pages.
    Les deux niveaux sont complementaires : un bon retrieval ne garantit pas
    une bonne generation, et vice versa.
    """
    questions      = result["parsed_output"].get("questions", [])
    retrieved      = result["retrieved_chunks"]
    expected_elem  = result.get("expected_answer_elements", [])
    relevant_pages = result.get("gold_pages", set())
    k              = result.get("k", number_of_text_chunks_to_retrieve)

    # Metriques de retrieval : independantes des questions generees
    prec = precision_at_k(retrieved, relevant_pages, k)
    rec  = recall_at_k(retrieved, relevant_pages, k)

    question_evals = []

    for q in questions:
        answer_text = q.get("answer", "")
        citations   = q.get("citations", [])

        # grounding : les mots-cles attendus sont-ils dans la reponse generee ?
        grounding = compute_grounding(answer_text, expected_elem)

        # elements_coverage : version detaillee pour le debug (found / missing)
        elem_cov, found_elems, missing_elems = compute_elements_coverage(
            answer_text, expected_elem
        )

        # citation_accuracy : les citations du LLM sont-elles traçables dans les sources ?
        cit_acc, valid_cit, total_cit = check_citations(citations, retrieved)

        question_evals.append({
            "question":          q.get("question", ""),
            "answer":            answer_text,
            "citations":         citations,
            "grounding_score":   grounding,
            "elements_coverage": elem_cov,
            "found_elements":    found_elems,
            "missing_elements":  missing_elems,
            "citation_accuracy": cit_acc,
            "valid_citations":   valid_cit,
            "total_citations":   total_cit,
        })

    n = len(question_evals)
    return {
        "query":            result["query"],
        "n_questions":      len(questions),
        # Retrieval
        "precision_at_k":   round(prec, 3),
        "recall_at_k":      round(rec, 3),
        # Generation (moyennes sur toutes les questions)
        "avg_grounding":    round(sum(q["grounding_score"]   for q in question_evals) / n, 3) if n else 0.0,
        "avg_citation_acc": round(sum(q["citation_accuracy"] for q in question_evals) / n, 3) if n else 0.0,
        "avg_elements_cov": round(sum(q["elements_coverage"] for q in question_evals) / n, 3) if n else 0.0,
        "question_evals":   question_evals,
        "refusal":          result.get("refusal"),
    }


# Evaluer toutes les requetes du gold set
generation_evals = [evaluate_generation(r) for r in results]

# Tableau de synthese avec les deux niveaux de metriques
print(f"{'Query':<45} | {'Prec@k':>6} | {'Rec@k':>6} | {'Ground':>6} | {'CitAcc':>6} | {'ElemCov':>7}")
print("-" * 95)
for ev in generation_evals:
    print(f"{ev['query'][:45]:<45} | "
          f"{ev['precision_at_k']:>6.3f} | "
          f"{ev['recall_at_k']:>6.3f} | "
          f"{ev['avg_grounding']:>6.3f} | "
          f"{ev['avg_citation_acc']:>6.3f} | "
          f"{ev['avg_elements_cov']:>7.3f}")

print("-" * 95)
n = len(generation_evals)
print(f"{'AVERAGE':<45} | "
      f"{sum(e['precision_at_k']   for e in generation_evals)/n:>6.3f} | "
      f"{sum(e['recall_at_k']      for e in generation_evals)/n:>6.3f} | "
      f"{sum(e['avg_grounding']    for e in generation_evals)/n:>6.3f} | "
      f"{sum(e['avg_citation_acc'] for e in generation_evals)/n:>6.3f} | "
      f"{sum(e['avg_elements_cov'] for e in generation_evals)/n:>7.3f}")

Query                                         | Prec@k |  Rec@k | Ground | CitAcc | ElemCov
-----------------------------------------------------------------------------------------------
What is the difference between a transformati |  0.000 |  0.000 |  0.000 |  0.000 |   0.000
Why is lazy evaluation important in Spark?    |  0.250 |  0.333 |  0.000 |  0.000 |   0.000
What is shuffle in Spark and why is it expens |  0.000 |  0.000 |  0.000 |  0.000 |   0.000
What problem does Lambda Architecture solve?  |  0.000 |  0.000 |  0.000 |  0.000 |   0.000
What is windowing in stream processing?       |  0.750 |  1.000 |  0.000 |  0.000 |   0.000
What is overfitting and why is it problematic |  0.000 |  0.000 |  0.000 |  0.000 |   0.000
What is bagging and why is it useful?         |  0.000 |  0.000 |  0.000 |  0.000 |   0.000
Why does lower perplexity not necessarily imp |  0.250 |  0.500 |  0.000 |  0.000 |   0.000
----------------------------------------------------------------------------

In [24]:
# Inspection detaillee du premier resultat pour comprendre ce qui manque.
# Cette cellule est utile quand on veut comprendre pourquoi une requete
# a un mauvais score de grounding ou de couverture d'elements.
ev = generation_evals[0]
print(f"Query: {ev['query']}\n")

for i, q_eval in enumerate(ev["question_evals"]):
    print(f"Question {i+1}: {q_eval['question'][:80]}")
    print(f"  Grounding    : {q_eval['grounding_score']:.3f}")
    print(f"  Citations OK : {q_eval['valid_citations']}/{q_eval['total_citations']}")
    print(f"  Found        : {q_eval['found_elements']}")
    print(f"  Missing      : {q_eval['missing_elements']}")
    print()

# On rappelle les elements attendus selon notre gold set pour faciliter la comparaison
print("Expected elements:")
for e in results[0]['expected_answer_elements']:
    print(f"  - {e}")


Query: What is the difference between a transformation and an action in Spark?

Expected elements:
  - transformation, RDD, immediately executing computation
  - An action triggers execution and produces a result or output


## Format-adherence metrics

In [25]:
# ================================================================
# SECTION 5 — Format-Adherence Metrics
# ================================================================

def evaluate_format_adherence(result: dict) -> dict:
    """
    Évalue si l'output LLM respecte le format JSON attendu.
    Métriques :
    - parse_success      : JSON parseable sans modification ?
    - required_fields    : tous les champs requis présents ?
    - citation_format    : citations au bon format ?
    - refusal_valid      : si refus, est-il bien formaté ?
    - overall_valid      : tout est valide ?
    """
    raw_output    = result["raw_output"]
    parsed_output = result["parsed_output"]
    parse_success = result["parse_success"]
    questions     = parsed_output.get("questions", [])
    refusal       = parsed_output.get("refusal")

    # ── 1. Parse success ──────────────────────────────────────
    # Déjà calculé dans le pipeline

    # ── 2. Required fields au niveau racine ───────────────────
    required_root_fields = {"query", "questions", "refusal"}
    present_root = set(parsed_output.keys())
    missing_root = required_root_fields - present_root
    root_fields_ok = len(missing_root) == 0

    # ── 3. Required fields par question ───────────────────────
    required_q_fields = {"question", "answer", "citations"}
    question_field_results = []

    for q in questions:
        present = set(q.keys())
        missing = required_q_fields - present
        question_field_results.append({
            "ok":     len(missing) == 0,
            "missing": list(missing)
        })

    questions_fields_ok = all(r["ok"] for r in question_field_results)
    questions_field_rate = (
        sum(1 for r in question_field_results if r["ok"]) / len(question_field_results)
        if question_field_results else 1.0
    )

    # ── 4. Citation format validity ───────────────────────────
    # Format attendu : contient pdf_name ou p.X ou chunk_id
    import re
    citation_pattern = re.compile(
        r"(\.pdf|p\.\d+|chunk_id|chunk|\[\d+\])", re.IGNORECASE
    )

    all_citations = []
    for q in questions:
        all_citations.extend(q.get("citations", []))

    valid_citation_formats = [
        bool(citation_pattern.search(str(c))) for c in all_citations
    ]
    citation_format_rate = (
        sum(valid_citation_formats) / len(valid_citation_formats)
        if valid_citation_formats else 1.0
    )

    # ── 5. Refusal format validity ────────────────────────────
    # Si refus : questions doit être vide et refusal doit être une string
    refusal_valid = True
    if refusal is not None:
        refusal_valid = (
            isinstance(refusal, str) and
            len(refusal) > 0 and
            len(questions) == 0
        )

    # ── 6. Overall validity ───────────────────────────────────
    overall_valid = (
        parse_success and
        root_fields_ok and
        questions_fields_ok and
        citation_format_rate >= 0.5
    )

    return {
        "query":                result["query"],
        "parse_success":        parse_success,
        "root_fields_ok":       root_fields_ok,
        "missing_root_fields":  list(missing_root),
        "questions_field_rate": round(questions_field_rate, 3),
        "citation_format_rate": round(citation_format_rate, 3),
        "n_citations":          len(all_citations),
        "refusal_valid":        refusal_valid,
        "overall_valid":        overall_valid,
    }


# Évaluer tous les résultats
format_evals = [evaluate_format_adherence(r) for r in results]

# ── Résumé ────────────────────────────────────────────────────
n_total   = len(format_evals)
n_parsed  = sum(1 for e in format_evals if e["parse_success"])
n_valid   = sum(1 for e in format_evals if e["overall_valid"])
n_root_ok = sum(1 for e in format_evals if e["root_fields_ok"])

print("=" * 60)
print("FORMAT-ADHERENCE METRICS")
print("=" * 60)
print(f"Parser pass rate        : {n_parsed}/{n_total} = {n_parsed/n_total:.1%}")
print(f"Overall valid rate      : {n_valid}/{n_total}  = {n_valid/n_total:.1%}")
print(f"Root fields OK          : {n_root_ok}/{n_total} = {n_root_ok/n_total:.1%}")
print(f"Avg question field rate : {sum(e['questions_field_rate'] for e in format_evals)/n_total:.1%}")
print(f"Avg citation format rate: {sum(e['citation_format_rate'] for e in format_evals)/n_total:.1%}")

# ── Détail par query ──────────────────────────────────────────
print(f"\n{'Query':<45} | {'Parse':>5} | {'Fields':>6} | {'Cite':>5} | {'Valid':>5}")
print("-" * 75)
for e in format_evals:
    print(f"{e['query'][:45]:<45} | "
          f"{'✅' if e['parse_success'] else '❌':>5} | "
          f"{e['questions_field_rate']:>6.1%} | "
          f"{e['citation_format_rate']:>5.1%} | "
          f"{'✅' if e['overall_valid'] else '❌':>5}")

FORMAT-ADHERENCE METRICS
Parser pass rate        : 0/8 = 0.0%
Overall valid rate      : 0/8  = 0.0%
Root fields OK          : 8/8 = 100.0%
Avg question field rate : 100.0%
Avg citation format rate: 100.0%

Query                                         | Parse | Fields |  Cite | Valid
---------------------------------------------------------------------------
What is the difference between a transformati |     ❌ | 100.0% | 100.0% |     ❌
Why is lazy evaluation important in Spark?    |     ❌ | 100.0% | 100.0% |     ❌
What is shuffle in Spark and why is it expens |     ❌ | 100.0% | 100.0% |     ❌
What problem does Lambda Architecture solve?  |     ❌ | 100.0% | 100.0% |     ❌
What is windowing in stream processing?       |     ❌ | 100.0% | 100.0% |     ❌
What is overfitting and why is it problematic |     ❌ | 100.0% | 100.0% |     ❌
What is bagging and why is it useful?         |     ❌ | 100.0% | 100.0% |     ❌
Why does lower perplexity not necessarily imp |     ❌ | 100.0% | 100.0% |     ❌

## System metrics


In [26]:
def compute_system_metrics(results: list) -> dict:
    """
    Agrège toutes les métriques système collectées pendant le pipeline.
    """
    n = len(results)
    if n == 0:
        return {}

    latencies_total    = [r["total_time_s"]       for r in results]
    latencies_retrieve = [r["retrieval_time_s"]    for r in results]
    latencies_generate = [r["generation_time_s"]   for r in results]
    tokens_in          = [r["tokens_in"]           for r in results]
    tokens_out         = [r["tokens_out"]          for r in results]
    memory_deltas      = [r["memory_delta_mb"]     for r in results]

    return {
        # Latence (secondes)
        "latency": {
            "mean_total_s":      round(sum(latencies_total)    / n, 3),
            "mean_retrieval_s":  round(sum(latencies_retrieve) / n, 3),
            "mean_generation_s": round(sum(latencies_generate) / n, 3),
            "min_total_s":       round(min(latencies_total), 3),
            "max_total_s":       round(max(latencies_total), 3),
        },
        # Tokens
        "tokens": {
            "mean_tokens_in":    round(sum(tokens_in)  / n, 1),
            "mean_tokens_out":   round(sum(tokens_out) / n, 1),
            "total_tokens_in":   sum(tokens_in),
            "total_tokens_out":  sum(tokens_out),
        },
        # Throughput
        "throughput": {
            "queries_per_minute": round(60 / (sum(latencies_total) / n), 2),
            "tokens_per_second":  round(
                sum(tokens_out) / sum(latencies_generate), 2
            ),
        },
        # Mémoire
        "memory": {
            "mean_delta_mb": round(sum(memory_deltas) / n, 1),
            "max_delta_mb":  round(max(memory_deltas), 1),
        },
        # Coût
        "cost": {
            "model":        "llama3.2:3b (local)",
            "monetary_usd": 0.0,
            "note":         "Local model — no API cost. Token and runtime costs reported above."
        },
        "n_queries": n,
    }


# Calculer les métriques
sys_metrics = compute_system_metrics(results)

# ── Affichage ─────────────────────────────────────────────────
print("=" * 55)
print("SYSTEM METRICS")
print("=" * 55)

print("\n── Latency ──────────────────────────────────────────")
print(f"  Mean total latency    : {sys_metrics['latency']['mean_total_s']}s")
print(f"  Mean retrieval time   : {sys_metrics['latency']['mean_retrieval_s']}s")
print(f"  Mean generation time  : {sys_metrics['latency']['mean_generation_s']}s")
print(f"  Min / Max total       : {sys_metrics['latency']['min_total_s']}s / {sys_metrics['latency']['max_total_s']}s")

print("\n── Tokens ───────────────────────────────────────────")
print(f"  Mean tokens in        : {sys_metrics['tokens']['mean_tokens_in']}")
print(f"  Mean tokens out       : {sys_metrics['tokens']['mean_tokens_out']}")
print(f"  Total tokens in       : {sys_metrics['tokens']['total_tokens_in']}")
print(f"  Total tokens out      : {sys_metrics['tokens']['total_tokens_out']}")

print("\n── Throughput ───────────────────────────────────────")
print(f"  Queries / minute      : {sys_metrics['throughput']['queries_per_minute']}")
print(f"  Tokens / second       : {sys_metrics['throughput']['tokens_per_second']}")

print("\n── Memory ───────────────────────────────────────────")
print(f"  Mean memory delta     : {sys_metrics['memory']['mean_delta_mb']} MB")
print(f"  Max memory delta      : {sys_metrics['memory']['max_delta_mb']} MB")

print("\n── Cost ─────────────────────────────────────────────")
print(f"  Model                 : {sys_metrics['cost']['model']}")
print(f"  Monetary cost         : ${sys_metrics['cost']['monetary_usd']}")
print(f"  Note                  : {sys_metrics['cost']['note']}")

print("\n── Per-query breakdown ──────────────────────────────")
print(f"{'Query':<45} | {'Total':>7} | {'Retr':>6} | {'Gen':>6} | {'Tok_in':>6} | {'Tok_out':>7}")
print("-" * 90)
for r in results:
    print(f"{r['query'][:45]:<45} | "
          f"{r['total_time_s']:>7.2f}s | "
          f"{r['retrieval_time_s']:>6.3f}s | "
          f"{r['generation_time_s']:>6.2f}s | "
          f"{r['tokens_in']:>6} | "
          f"{r['tokens_out']:>7}")

SYSTEM METRICS

── Latency ──────────────────────────────────────────
  Mean total latency    : 34.019s
  Mean retrieval time   : 7.482s
  Mean generation time  : 26.538s
  Min / Max total       : 28.783s / 43.758s

── Tokens ───────────────────────────────────────────
  Mean tokens in        : 269.4
  Mean tokens out       : 163.2
  Total tokens in       : 2155
  Total tokens out      : 1306

── Throughput ───────────────────────────────────────
  Queries / minute      : 1.76
  Tokens / second       : 6.15

── Memory ───────────────────────────────────────────
  Mean memory delta     : 0.0 MB
  Max memory delta      : 0.0 MB

── Cost ─────────────────────────────────────────────
  Model                 : llama3.2:3b (local)
  Monetary cost         : $0.0
  Note                  : Local model — no API cost. Token and runtime costs reported above.

── Per-query breakdown ──────────────────────────────
Query                                         |   Total |   Retr |    Gen | Tok_in | T

## Regression testing


In [27]:
# Ces prompts ne changent jamais. Ils permettent de detecter rapidement si un changement
# de configuration (modele, embedding, k...) degrade les resultats sur des questions connues.
REGRESSION_PROMPTS = [
    "What is MapReduce and how does it work?",
    "How does HDFS handle fault tolerance?",
    "What is the CAP theorem?",
    "Explain RDD lineage in Spark.",
    "How does LSH work for similarity search?",
]

def run_regression_test(
    prompts: list,
    embedding_strategy: str,
    similarity_metric: str,
    k: int = 5,
    n_questions: int = 2,
    model: str = "llama3.2:3b",
    save_path: str = "/home/jovyan/work/regression_baseline.json",
) -> dict:
    """
    Lance le pipeline complet sur les prompts fixes et enregistre les resultats.
    Si une baseline existe deja sur disque, on compare avec elle pour detecter
    toute regression ou amelioration par rapport a la configuration precedente.
    """
    print(f"Running {len(prompts)} regression tests with {embedding_strategy}/{similarity_metric}...")
    
    current_run = {
        "timestamp":          datetime.now().isoformat(),
        "embedding_strategy": embedding_strategy,
        "similarity_metric":  similarity_metric,
        "k":                  k,
        "model":              model,
        "results":            {}
    }
    
    for prompt in prompts:
        print(f"  Traitement : {prompt[:60]}...")
        # On reutilise le meme pipeline que pour l'evaluation principale
        result = run_generation_pipeline(
            query=prompt,
            n_questions=n_questions,
        )
        
        # On ne garde que les metriques essentielles pour alleger le fichier de baseline
        current_run["results"][prompt] = {
            "parse_success":  result["parse_success"],
            "n_questions":    result["n_questions"],
            "tokens_in":      result["tokens_in"],
            "tokens_out":     result["tokens_out"],
            "total_time_s":   result["total_time_s"],
            "questions":      result["parsed_output"].get("questions", []),
            "refusal":        result["parsed_output"].get("refusal"),
        }
    
    return current_run


def compare_with_baseline(current_run: dict, save_path: str) -> dict:
    """
    Compare le run actuel avec la baseline sauvegardee sur disque.
    Si aucune baseline n'existe encore, on sauvegarde ce run comme reference.
    """
    path = Path(save_path)
    
    # Premier lancement : pas encore de baseline, on sauvegarde et on s'arrete la
    if not path.exists():
        with open(path, "w") as f:
            json.dump(current_run, f, indent=2)
        print(f"Baseline sauvegardee dans {save_path}")
        return {"status": "baseline_created", "comparisons": []}
    
    # Une baseline existe : on charge et on compare les metriques cles
    with open(path) as f:
        baseline = json.load(f)
    
    print(f"Baseline du : {baseline['timestamp']}")
    print(f"Run actuel  : {current_run['timestamp']}")
    
    comparisons  = []
    regressions  = []
    improvements = []
    
    for prompt in current_run["results"]:
        if prompt not in baseline["results"]:
            print(f"  Nouveau prompt (absent de la baseline) : {prompt[:50]}")
            continue
        
        curr = current_run["results"][prompt]
        base = baseline["results"][prompt]
        
        # Critere 1 : le parsing JSON doit rester stable
        parse_ok     = curr["parse_success"] == base["parse_success"]
        # Critere 2 : on tolere 1 question de moins (le LLM peut varier un peu)
        n_q_ok       = curr["n_questions"] >= base["n_questions"] - 1
        time_delta   = curr["total_time_s"]  - base["total_time_s"]
        tokens_delta = curr["tokens_out"]    - base["tokens_out"]

        status = "OK"
        if not parse_ok:
            status = "REGRESSION : echec du parsing JSON"
            regressions.append(prompt)
        elif not n_q_ok:
            status = "REGRESSION : moins de questions generees"
            regressions.append(prompt)
        elif curr["n_questions"] > base["n_questions"]:
            status = "AMELIORATION : plus de questions generees"
            improvements.append(prompt)

        comparisons.append({
            "prompt":       prompt,
            "status":       status,
            "parse":        f"{'OK' if curr['parse_success'] else 'FAIL'} (was {'OK' if base['parse_success'] else 'FAIL'})",
            "n_questions":  f"{curr['n_questions']} (was {base['n_questions']})",
            "time_delta_s": round(time_delta, 2),
            "tokens_delta": tokens_delta,
        })
    
    return {
        "status":          "compared",
        "n_regressions":   len(regressions),
        "n_improvements":  len(improvements),
        "n_stable":        len(comparisons) - len(regressions) - len(improvements),
        "regressions":     regressions,
        "improvements":    improvements,
        "comparisons":     comparisons,
    }


def display_regression_report(comparison: dict) -> None:
    """Affiche un rapport lisible de la comparaison baseline vs run actuel."""
    
    if comparison["status"] == "baseline_created":
        print("Baseline creee. Relancez apres un changement pour voir les regressions.")
        return
    
    print("\n" + "=" * 60)
    print("RAPPORT DE REGRESSION")
    print("=" * 60)
    print(f"  Regressions  : {comparison['n_regressions']}")
    print(f"  Ameliorations: {comparison['n_improvements']}")
    print(f"  Stables      : {comparison['n_stable']}")
    
    print(f"\n{'Prompt':<45} | {'Statut':<35} | {'Questions':>9} | {'Delta t':>7}")
    print("-" * 105)
    for c in comparison["comparisons"]:
        print(f"{c['prompt'][:45]:<45} | "
              f"{c['status']:<35} | "
              f"{c['n_questions']:>9} | "
              f"{c['time_delta_s']:>+7.1f}s")
    
    if comparison["n_regressions"] > 0:
        print(f"\n{comparison['n_regressions']} regression(s) detectee(s) :")
        for r in comparison["regressions"]:
            print(f"  -> {r}")
    else:
        print("\nAucune regression detectee.")


# Lancer les tests de regression avec la configuration actuelle du notebook
REGRESSION_SAVE_PATH = "/home/jovyan/work/regression_baseline.json"

current_run = run_regression_test(
    prompts=REGRESSION_PROMPTS,
    embedding_strategy=embedding_strategy,
    similarity_metric=similarity_strategy,
    k=number_of_text_chunks_to_retrieve,
    n_questions=2,
    model=generator_model,
    save_path=REGRESSION_SAVE_PATH,
)

comparison = compare_with_baseline(current_run, REGRESSION_SAVE_PATH)
display_regression_report(comparison)


Running 5 regression tests with tfidf/cosine...
  Traitement : What is MapReduce and how does it work?...
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


  Traitement : How does HDFS handle fault tolerance?...
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


  Traitement : What is the CAP theorem?...
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


  Traitement : Explain RDD lineage in Spark....
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


  Traitement : How does LSH work for similarity search?...
🔍 Recherche en cours... [Stratégie: tfidf | Métrique: cosine]


FileNotFoundError: [Errno 2] No such file or directory: '/home/jovyan/work/regression_baseline.json'

## Manual evaluation


In [ ]:
# ================================================================
# SECTION 8 — Evaluation manuelle de la qualite des questions
# ================================================================

# Grille de notation : 0 = mauvais, 1 = acceptable, 2 = bon
RUBRIC = {
    "relevance":        "La question est-elle liee au topic et aux chunks recuperes ?",
    "correctness":      "Est-elle techniquement correcte par rapport au cours ?",
    "faithfulness":     "Est-elle basee sur les chunks, sans hallucination ?",
    "citation_support": "Les citations pointent-elles vers les bons chunks ?",
    "clarity":          "La question est-elle bien formulee pour un examen ?",
    "format_adherence": "L'output suit-il le format requis (JSON, champs presents) ?",
}

def display_for_manual_evaluation(results: list, n_samples: int = 5) -> None:
    """
    Affiche les questions generees pour permettre une notation manuelle.
    On selectionne les n_samples premiers resultats (ou tous si moins nombreux).
    Cette etape est importante car les metriques automatiques ne capturent pas
    tout : la clarte, la pertinence pedagogique, l'absence d'hallucination subtile.
    """
    samples = results[:n_samples]
    
    for r_idx, result in enumerate(samples):
        print(f"\n{'='*70}")
        print(f"[{r_idx+1}/{len(samples)}] QUERY: {result['query']}")
        print(f"Embedding: {result['embedding_strategy']} | "
              f"Metric: {result['similarity_metric']} | k={result['k']}")
        
        # Affichage des chunks recuperes (les 3 premiers suffisent pour la relecture)
        print(f"\nChunks recuperes :")
        for i, row in enumerate(result["retrieved_chunks"][:3]):
            # Les rows Spark ont des colonnes directes, pas de sous-dict
            print(f"  [{i+1}] {row['source_pdf']} p.{row['page_num']} | "
                  f"score={row['score']:.3f}")
            print(f"       {row['chunk_text'][:100]}...")
        
        print(f"\nQuestions generees :")
        questions = result["parsed_output"].get("questions", [])
        if not questions:
            print("  (aucune question generee — le modele a peut-etre refuse ou le JSON est invalide)")
        for q_idx, q in enumerate(questions):
            print(f"\n  Q{q_idx+1}: {q['question']}")
            print(f"  A : {q['answer'][:200]}...")
            print(f"  Citations: {q['citations']}")
        
        # Grille vierge a remplir manuellement apres lecture
        print(f"\n{'─'*70}")
        print("GRILLE DE NOTATION (0=mauvais, 1=acceptable, 2=bon) :")
        print(f"  {'Critere':<20} | {'Note':>5} | Description")
        print(f"  {'-'*65}")
        for criterion, description in RUBRIC.items():
            print(f"  {criterion:<20} | {'?/2':>5} | {description}")


# Afficher les resultats pour la relecture manuelle
display_for_manual_evaluation(results, n_samples=min(5, len(results)))


In [ ]:
# Noté manuellement :
MANUAL_SCORES = [
    {
        "query":    "What is Big Data and what are its main characteristics?",
        "question": "What are the three main characteristics of Big Data?",
        "scores": {
            "relevance":        2,   # ← remplir 0, 1, ou 2
            "correctness":      2,
            "faithfulness":     2,
            "citation_support": 1,
            "clarity":          2,
            "format_adherence": 2,
        },
        "notes": "Good question, answer slightly incomplete on scalability aspect."
    },
    # Ajouter les autres questions ici...
]

def compute_manual_scores_summary(manual_scores: list) -> None:
    """Affiche le résumé des scores manuels."""
    
    if not manual_scores:
        print("No manual scores yet.")
        return
    
    criteria = list(RUBRIC.keys())
    
    print("\n" + "=" * 70)
    print("MANUAL QUALITY EVALUATION SUMMARY")
    print("=" * 70)
    
    # Score par critère
    print(f"\n{'Criterion':<20} | {'Mean':>6} | {'Min':>4} | {'Max':>4} | {'%Good (=2)':>10}")
    print("-" * 60)
    
    for criterion in criteria:
        scores = [s["scores"][criterion] for s in manual_scores 
                  if criterion in s["scores"]]
        if not scores:
            continue
        mean  = sum(scores) / len(scores)
        pct_good = sum(1 for s in scores if s == 2) / len(scores)
        print(f"  {criterion:<18} | {mean:>6.2f} | {min(scores):>4} | "
              f"{max(scores):>4} | {pct_good:>10.1%}")
    
    # Score global par question
    print(f"\n{'Question':<50} | {'Total':>5} | {'%':>5}")
    print("-" * 65)
    for item in manual_scores:
        total = sum(item["scores"].values())
        max_score = len(item["scores"]) * 2
        pct = total / max_score
        print(f"  {item['question'][:48]:<48} | {total:>4}/{max_score} | {pct:>5.1%}")
        if item.get("notes"):
            print(f"    Note: {item['notes']}")
    
    # Moyenne globale
    all_totals = [sum(s["scores"].values()) for s in manual_scores]
    max_total  = len(criteria) * 2
    global_mean = sum(all_totals) / len(all_totals)
    print(f"\n  Overall mean score : {global_mean:.2f}/{max_total} "
          f"({global_mean/max_total:.1%})")


compute_manual_scores_summary(MANUAL_SCORES)